In [ ]:
import os
import json
from zipfile import ZipFile
import pandas as pd
from langchain_groq import ChatGroq

DATASET = "/content/archive.zip"


def load_results(path=DATASET):
    if path.endswith(".zip"):
        with ZipFile(path) as z:
            return pd.read_csv(z.open("results.csv"), parse_dates=["date"])
    return pd.read_csv(path, parse_dates=["date"])


df = load_results()


def form(team, n=5):
    matches = (
        df[(df.home_team == team) | (df.away_team == team)]
        .sort_values("date")
        .tail(n)
    )

    result = []

    for _, row in matches.iterrows():
        if row.home_team == team:
            scored = row.home_score
            conceded = row.away_score
        else:
            scored = row.away_score
            conceded = row.home_score

        if scored > conceded:
            result.append("W")
        elif scored == conceded:
            result.append("D")
        else:
            result.append("L")

    return "-".join(result) if result else "N/A"


def stats(team):
    matches = df[(df.home_team == team) | (df.away_team == team)].sort_values(
        "date"
    )

    if len(matches) == 0:
        return None

    wins = draws = losses = 0
    gf = ga = 0

    for _, row in matches.iterrows():
        if row.home_team == team:
            scored = row.home_score
            conceded = row.away_score
        else:
            scored = row.away_score
            conceded = row.home_score

        gf += scored
        ga += conceded

        if scored > conceded:
            wins += 1
        elif scored == conceded:
            draws += 1
        else:
            losses += 1

    return {
        "matches": len(matches),
        "wins": wins,
        "draws": draws,
        "losses": losses,
        "goals_for": gf,
        "goals_against": ga,
        "avg_goals": round(gf / len(matches), 2),
        "avg_conceded": round(ga / len(matches), 2),
        "recent_form": form(team),
    }


def h2h(team_a, team_b):
    matches = df[
        ((df.home_team == team_a) & (df.away_team == team_b))
        | ((df.home_team == team_b) & (df.away_team == team_a))
    ]

    a = b = d = 0

    for _, row in matches.iterrows():
        if row.home_score == row.away_score:
            d += 1

        elif (
            row.home_team == team_a and row.home_score > row.away_score
        ) or (row.away_team == team_a and row.away_score > row.home_score):
            a += 1

        else:
            b += 1

    return {"team_a_wins": a, "draws": d, "team_b_wins": b}


def predict(
    team_a, team_b, stage="Group Stage", model="llama-3.3-70b-versatile"
):
    team_a_stats = stats(team_a)
    team_b_stats = stats(team_b)

    if team_a_stats is None or team_b_stats is None:
        raise ValueError("Unknown team.")

    head2head = h2h(team_a, team_b)

    llm = ChatGroq(
        model=model,
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0.2,
    )

    prompt = f"""
You are a professional football analyst.

Predict the FIFA World Cup 2026 match.

Stage:
{stage}

Team A:
{team_a}

Statistics:
{json.dumps(team_a_stats, indent=2)}

Team B:
{team_b}

Statistics:
{json.dumps(team_b_stats, indent=2)}

Head-to-head:
{json.dumps(head2head, indent=2)}

Use:
- Historical performance
- Recent form
- Goals scored
- Goals conceded
- Head-to-head
- Overall team quality

Return ONLY valid JSON.

{{
    "predicted_winner":"",
    "confidence_percentage":0,
    "key_reasoning":""
}}
"""

    response = llm.invoke(
        [
            (
                "system",
                "You are an expert football analyst. Return ONLY JSON.",
            ),
            ("human", prompt),
        ]
    )

    content = response.content.strip()

    if content.startswith("```"):
        content = (
            content.replace("```json", "").replace("```", "").strip()
        )

    prediction = json.loads(content)

    return prediction


if __name__ == "__main__":
    # Replace with your own NEW Groq API key or set it as an environment variable.
    os.environ["GROQ_API_KEY"] = (
        ""
    )

    result = predict("Paraguay", "France")

    print("\n==============================")
    print(" FIFA WORLD CUP 2026 PREDICTION")
    print("==============================")
    print(f"Winner      : {result['predicted_winner']}")
    print(f"Confidence  : {result['confidence_percentage']}%")
    print(f"Reason      : {result['key_reasoning']}")


 FIFA WORLD CUP 2026 PREDICTION
Winner      : France
Confidence  : 70%
Reason      : France has a stronger historical performance with more wins and a higher win percentage. Their recent form is also superior, with four consecutive wins before a loss, indicating a high level of current performance. The head-to-head record further favors France, with four wins and only two draws against Paraguay. Although goals scored and conceded are not provided, France's overall team quality and recent form suggest they are more likely to win.


In [ ]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.8 MB/s eta 0:00:00


In [ ]:
import gradio as gr

teams = sorted(
    set(df["home_team"]).union(set(df["away_team"]))
)

stages = [
    "Group Stage",
    "Round of 32",
    "Round of 16",
    "Quarter Final",
    "Semi Final",
    "Third Place",
    "Final"
]


def predict_ui(team1, team2, stage):

    result = predict(team1, team2, stage)

    return (
        result["predicted_winner"],
        str(result["confidence_percentage"]) + "%",
        result["key_reasoning"],
    )


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# ⚽ FIFA World Cup 2026 Predictor")
    gr.Markdown("Select two teams and let the AI predict the winner.")

    with gr.Row():
        team1 = gr.Dropdown(
            teams,
            value="Brazil",
            label="Team A"
        )

        team2 = gr.Dropdown(
            teams,
            value="Argentina",
            label="Team B"
        )

    stage = gr.Dropdown(
        stages,
        value="Group Stage",
        label="Stage"
    )

    btn = gr.Button("Predict Winner", variant="primary")

    winner = gr.Textbox(label="🏆 Predicted Winner")
    confidence = gr.Textbox(label="📈 Confidence")
    reason = gr.Textbox(
        label="🧠 Reason",
        lines=5
    )

    btn.click(
        predict_ui,
        inputs=[team1, team2, stage],
        outputs=[winner, confidence, reason]
    )

demo.launch()

/tmp/ipykernel_693/3054772547.py:29: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://615a5e0fd342632ed3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.7 MB/s eta 0:00:00


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

teams = sorted(set(df["home_team"]).union(set(df["away_team"])))

team1 = widgets.Dropdown(
    options=teams,
    value="Brazil",
    description="Team A:"
)

team2 = widgets.Dropdown(
    options=teams,
    value="Argentina",
    description="Team B:"
)

stage = widgets.Dropdown(
    options=[
        "Group Stage",
        "Round of 16",
        "Quarter Final",
        "Semi Final",
        "Final"
    ],
    value="Group Stage",
    description="Stage:"
)

button = widgets.Button(description="Predict", button_style="success")
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        result = predict(team1.value, team2.value, stage.value)

        print("🏆 Winner:", result["predicted_winner"])
        print("📊 Confidence:", result["confidence_percentage"], "%")
        print("💡 Reason:", result["key_reasoning"])

button.on_click(on_click)

display(team1, team2, stage, button, output)

Dropdown(description='Team A:', index=39, options=('Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria'…

Dropdown(description='Team B:', index=13, options=('Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria'…

Dropdown(description='Stage:', options=('Group Stage', 'Round of 16', 'Quarter Final', 'Semi Final', 'Final'),…

Button(button_style='success', description='Predict', style=ButtonStyle())

Output()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

teams = sorted(set(df["home_team"]).union(set(df["away_team"])))

# ---------- Title ----------
title = widgets.HTML("""
<div style="
background:linear-gradient(90deg,#1e3c72,#2a5298);
padding:20px;
border-radius:12px;
text-align:center;
color:white;
font-size:30px;
font-weight:bold;
">
🏆 FIFA World Cup 2026 AI Predictor ⚽
</div>
""")

subtitle = widgets.HTML("""
<div style="
text-align:center;
font-size:16px;
color:#666;
margin:10px 0 20px 0;
">
Select two teams and let AI predict the winner.
</div>
""")

# ---------- Dropdowns ----------
style = {"description_width": "90px"}

team1 = widgets.Dropdown(
    options=teams,
    value="Brazil",
    description="🏠 Team A",
    layout=widgets.Layout(width="350px"),
    style=style
)

team2 = widgets.Dropdown(
    options=teams,
    value="Argentina",
    description="✈️ Team B",
    layout=widgets.Layout(width="350px"),
    style=style
)

stage = widgets.Dropdown(
    options=[
        "Group Stage",
        "Round of 16",
        "Quarter Final",
        "Semi Final",
        "Final"
    ],
    value="Group Stage",
    description="🏟️ Stage",
    layout=widgets.Layout(width="350px"),
    style=style
)

# ---------- Button ----------
button = widgets.Button(
    description=" Predict Winner",
    icon="soccer-ball-o",
    button_style="success",
    layout=widgets.Layout(width="220px", height="45px")
)

output = widgets.Output(
    layout=widgets.Layout(
        border="2px solid #2a5298",
        padding="20px",
        margin="20px 0 0 0"
    )
)

# ---------- Prediction ----------
def on_click(b):

    with output:
        clear_output()

        try:
            result = predict(team1.value, team2.value, stage.value)

            display(widgets.HTML(f"""
            <div style="
            background:#f8f9fa;
            padding:25px;
            border-radius:15px;
            border-left:8px solid #2a5298;
            ">

            <h2 style="color:#2a5298;">🏆 Prediction Result</h2>

            <p style="font-size:20px;">
            <b>Winner:</b>
            <span style="color:green;">
            {result['predicted_winner']}
            </span>
            </p>

            <p style="font-size:18px;">
            <b>Confidence:</b>
            {result['confidence_percentage']}%
            </p>

            <hr>

            <h4>🧠 AI Reasoning</h4>

            <p style="font-size:16px; line-height:1.6;">
            {result['key_reasoning']}
            </p>

            </div>
            """))

        except Exception as e:
            display(widgets.HTML(f"""
            <div style="
            background:#ffe6e6;
            padding:20px;
            border-radius:10px;
            color:red;
            ">
            ❌ <b>Error:</b> {e}
            </div>
            """))

button.on_click(on_click)

# ---------- Layout ----------
left = widgets.VBox(
    [team1, team2, stage],
    layout=widgets.Layout(
        padding="10px",
        align_items="center"
    )
)

center = widgets.HBox(
    [button],
    layout=widgets.Layout(justify_content="center", margin="20px")
)

ui = widgets.VBox([
    title,
    subtitle,
    left,
    center,
    output
])

display(ui)

In [ ]:
import gradio as gr

# Assumes you already defined:
# - df
# - predict(team_a, team_b, stage)

teams = sorted(set(df["home_team"]).union(set(df["away_team"])))
stages = ["Group Stage","Round of 16","Quarter Final","Semi Final","Final"]

CSS = """
body{background:#07111f;}
.gradio-container{max-width:960px!important}
.hero{
background:linear-gradient(135deg,#0f172a,#1d4ed8);
color:white;padding:22px;border-radius:18px;
text-align:center;margin-bottom:16px;
}
.card{
background:rgba(255,255,255,.05);
border:1px solid rgba(255,255,255,.12);
border-radius:18px;
padding:18px;
}
.footer{text-align:center;color:#aaa;margin-top:15px}
"""

HEAD = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Poppins:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet"
href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.6.0/css/all.min.css">
<style>
*{font-family:Poppins,sans-serif}
</style>
"""

def ui_predict(a,b,stage):
    if a==b:
        return "Choose different teams","",""
    r = predict(a,b,stage)
    return (
        "🏆 "+r["predicted_winner"],
        str(r["confidence_percentage"])+"%",
        r["key_reasoning"]
    )

with gr.Blocks(css=CSS, head=HEAD, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class="hero">
        <h1>🏆 FIFA World Cup 2026 AI Predictor</h1>
        <p><i class="fa-solid fa-futbol"></i>
        Historical data + Groq Llama Prediction</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(elem_classes="card"):
            team1 = gr.Dropdown(teams,value="Brazil",label="⚽ Team A")
            team2 = gr.Dropdown(teams,value="Argentina",label="⚽ Team B")
            stage = gr.Dropdown(stages,value="Group Stage",label="🏟 Stage")
            btn = gr.Button("🚀 Predict",variant="primary")
        with gr.Column(elem_classes="card"):
            winner = gr.Textbox(label="🏆 Winner")
            confidence = gr.Textbox(label="📈 Confidence")
            reason = gr.Textbox(label="🧠 AI Reasoning",lines=8)

    btn.click(ui_predict,[team1,team2,stage],[winner,confidence,reason])

    gr.HTML('<div class="footer">Powered by Gradio + Groq + Historical Match Data</div>')

if __name__ == "__main__":
    demo.launch()


/tmp/ipykernel_4647/2499976646.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css, head. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, head=HEAD, theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
